# 06 · vLLM disaggregated prefill/decode: ScuffedRdmaConnector TTFT

KV cache transfer for disaggregated prefill (Cerberus) / decode (Chimera) through `ScuffedRdmaConnector`
(`middleware/vllm_plugin/`), compared against vLLM's `NixlConnector`, the same connector over TCP, and a single
non-disaggregated server.

- **local**: runs the connector test suite, then pushes a Granite 3.3-2B shaped KV cache between two processes
  through the connector's own data path (no vLLM, no GPU), over TCP and over the libfabric pool when it is built.
  Numbers from this mode are smoke tests of the code path, not results.
- **cluster**: prints the serve/proxy commands for each setup (start them in terminals on each node), checks the
  cross-node transport with the loopback tool, runs the TTFT/ITL sweep against every endpoint, and plots p50/p99
  against prompt length. Results land in `benchmarks/results/vllm_disagg/`.

In [ ]:
MODE = "local"            # "local": tests + loopback on this machine; "cluster": real vLLM on Cerberus/Chimera
PREFILL_HOST = "cerberus"
DECODE_HOST = "chimera"
MODEL = "ibm-granite/granite-3.3-2b-instruct"
TRANSPORT = "fabric"       # fabric | tcp
PROVIDER = "verbs"         # libfabric provider for TRANSPORT="fabric" (verbs on SoftRoCE; tcp/shm work anywhere)
EP_TYPE = "msg"            # msg | rdm
RESULTS_SUBDIR = "vllm_disagg"
KV_PORT = 14600
# cluster-mode sweep targets: label -> base URL (proxy for disagg setups, server for "single")
ENDPOINTS = {
    "single": "http://localhost:8300",
    "nixl": "http://localhost:8001",
    "scuffed-tcp": "http://localhost:8002",
    "scuffed-fabric-verbs": "http://localhost:8000",
}
PROMPT_LENS = [128, 512, 1024, 2048, 4096]
N_PER_LEN = 20
MAX_TOKENS = 64
WAIT_FOR_SERVERS_S = 0      # >0: poll /health on every endpoint before sweeping

In [ ]:
import os, sys, json, socket, subprocess, time
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "middleware").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
HOST = socket.gethostname()
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
RESULTS = (ROOT / "benchmarks" / "results" / RESULTS_SUBDIR) if MODE == "cluster" else Path("/tmp/scuffed-local") / RESULTS_SUBDIR
RESULTS.mkdir(parents=True, exist_ok=True)
def save(name, obj):
    p = RESULTS / f"{HOST}_{name}.json"
    obj = {"host": HOST, "utc": STAMP, "mode": MODE, **obj}
    p.write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", p)
    return p
def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=ROOT)
    print(r.stdout[-4000:], r.stderr[-2000:])
    if check and r.returncode:
        raise RuntimeError(f"{cmd} -> {r.returncode}")
    return r
print("repo:", ROOT, "| host:", HOST, "| mode:", MODE, "| results ->", RESULTS)

## 1. Connector tests

Scheduler bookkeeping, block gather/scatter for every KV layout, codec (bfloat16, scuffedQuant), producer to consumer through the real connector class, proxy, TTFT runner. The two vLLM-import tests run when vLLM is installed (cluster) or its source is on `VLLM_SRC`.

In [ ]:
r = sh("python3 -m pytest -q -rs -p no:warnings tests/test_vllm_connector.py", check=False)
assert r.returncode == 0, "connector tests failed"

## 2. Loopback through the connector data path

`take_blocks -> encode_kv -> KVSender -> BlobChannel -> KVReceiver -> decode_kv -> put_blocks`, Granite 3.3-2B cache shape (40 layers, 8 KV heads, head_dim 64, bfloat16), FlashAttention layout. Consumer runs in a child process. Every request is checked by SHA-256 on both sides.

In [ ]:
from middleware.vllm_plugin.loopback import default_args, run_both
if MODE == "local":
    lb_lens, lb_layers = [128, 512], 8
else:
    lb_lens, lb_layers = [128, 512, 2048, 4096], 40
loop = {}
for tr, prov in [("tcp", None), ("fabric", "tcp" if MODE == "local" else PROVIDER)]:
    out = run_both(default_args(layers=lb_layers, prompt_lens=lb_lens, transport=tr, provider=prov,
                                ep_type=EP_TYPE if tr == "fabric" else None, num_blocks=512))
    used = (out["producer"]["transport"], out["consumer"]["transport"])
    loop[f"{tr}:{prov}"] = out
    print(f"{tr:6s} provider={prov} used={used} all_match={out['all_match']}")
    for p, c in zip(out["producer"]["requests"], out["consumer"]["requests"]):
        print(f"   prompt {p['prompt_len']:5d}  push {p['push_ms']:8.2f} ms  load {c['load_ms']:8.2f} ms  "
              f"blocks {c['blocks_written']}")
    assert out["all_match"], "KV mismatch after transfer"
save("loopback", {"note": "same-host two-process loopback; smoke test of the connector data path",
                  "runs": {k: {"producer": v["producer"]["requests"], "consumer": v["consumer"]["requests"],
                               "transport": [v["producer"]["transport"], v["consumer"]["transport"]],
                               "producer_telemetry": v["producer"]["telemetry"]} for k, v in loop.items()}})

### Telemetry from the loopback producer

Per-class (hot/cold) ops, bytes and completion latency as reported to `middleware.telemetry` by `KVSender`. KV layers land in cold, the `__done__` control blobs in hot.

In [ ]:
rows = []
for k, v in loop.items():
    for cls, s in v["producer"]["telemetry"]["classes"].items():
        rows.append({"run": k, "class": cls, "ops": s["ops"], "bytes": s["bytes"],
                     "p50_us": s["p50_us"], "p99_us": s["p99_us"]})
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)

## 3. Cluster: start the servers

Run each block in a terminal on the named node (decode first). Each setup uses its own ports so they can stay up together if memory allows; otherwise run one setup at a time and edit `ENDPOINTS`.

Cross-node transport check without vLLM (run before the servers):

```
chimera$  python -m middleware.vllm_plugin.loopback --role consumer --port 14600 --transport fabric --provider verbs
cerberus$ python -m middleware.vllm_plugin.loopback --role producer --peer chimera --port 14600 --transport fabric --provider verbs
```

In [ ]:
setups = {
    "scuffed-fabric-verbs": dict(CONNECTOR="scuffed", TRANSPORT=TRANSPORT, PROVIDER=PROVIDER, EP_TYPE=EP_TYPE,
                                 PROXY_PORT=8000, KV_PORT=KV_PORT),
    "scuffed-tcp": dict(CONNECTOR="tcp", PROXY_PORT=8002, KV_PORT=KV_PORT + 100, PREFILL_PORT=8102, DECODE_PORT=8202),
    "nixl": dict(CONNECTOR="nixl", PROXY_PORT=8001, PREFILL_PORT=8101, DECODE_PORT=8201),
}
for label, env in setups.items():
    env = dict(env, MODEL=MODEL, PREFILL_HOST=PREFILL_HOST, DECODE_HOST=DECODE_HOST)
    pre = " ".join(f"{k}={v}" for k, v in env.items())
    print(f"# ---- {label}")
    print(sh(f"{pre} bash benchmarks/vllm_disagg/launch.sh print", check=False).stdout)

In [ ]:
import urllib.request
def healthy(url):
    for path in ("/health", "/healthcheck"):
        try:
            if urllib.request.urlopen(url + path, timeout=3).status == 200:
                return True
        except Exception:
            pass
    return False

if MODE == "cluster" and WAIT_FOR_SERVERS_S > 0:
    deadline = time.time() + WAIT_FOR_SERVERS_S
    pending = dict(ENDPOINTS)
    while pending and time.time() < deadline:
        pending = {k: u for k, u in pending.items() if not healthy(u)}
        time.sleep(5)
    print("not reachable:", list(pending) or "none")

## 4. TTFT / ITL sweep

Random token-id prompts of exact length, `ignore_eos`, streaming. In local mode the runner measures a mock server with known delays (40 ms to first token, 5 ms between tokens) to check the measurement path.

In [ ]:
sys.path.insert(0, str(ROOT / "benchmarks" / "vllm_disagg"))
import importlib, run_ttft_sweep
importlib.reload(run_ttft_sweep)
sweeps = {}
if MODE == "local":
    sweeps["selftest-mock"] = run_ttft_sweep.selftest()
else:
    for label, url in ENDPOINTS.items():
        if not healthy(url):
            print("skip", label, "(not reachable)")
            continue
        res = run_ttft_sweep.sweep(url, MODEL, PROMPT_LENS, N_PER_LEN, MAX_TOKENS)
        res["label"] = label
        sweeps[label] = res
        save(f"ttft_{label}", res)
list(sweeps)

## 5. Plot

TTFT and ITL percentiles against prompt length for every setup measured.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, res in sweeps.items():
    L = [r["prompt_len"] for r in res["results"]]
    for ax, key in ((axes[0], "ttft"), (axes[1], "itl")):
        p50 = [r[f"{key}_p50_ms"] for r in res["results"]]
        p99 = [r[f"{key}_p99_ms"] for r in res["results"]]
        line, = ax.plot(L, p50, marker="o", label=f"{label} p50")
        ax.plot(L, p99, marker="x", linestyle="--", color=line.get_color(), label=f"{label} p99")
for ax, t in zip(axes, ("TTFT (ms)", "ITL (ms)")):
    ax.set_xscale("log", base=2); ax.set_xlabel("prompt tokens"); ax.set_ylabel(t); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
title = "mock server (measurement check, not results)" if MODE == "local" else f"{MODEL}: {PREFILL_HOST} prefill, {DECODE_HOST} decode"
fig.suptitle(title)
plt.tight_layout()
fig.savefig(RESULTS / f"{HOST}_ttft_itl.png", dpi=120)
plt.show()

## 6. Connector telemetry from the running servers

With `metrics_port` set in `kv_connector_extra_config` (launch.sh sets 9464), each worker serves Prometheus text on `metrics_port + rank`: per-class ops, bytes, QP error/reconnect/TCP-fallback events, completion latency histogram.

In [ ]:
if MODE == "cluster":
    for host in (PREFILL_HOST, DECODE_HOST):
        try:
            txt = urllib.request.urlopen(f"http://{host}:9464/metrics", timeout=5).read().decode()
            print(f"# {host}"); print("\n".join(l for l in txt.splitlines() if not l.startswith("#"))[:3000])
        except Exception as e:
            print(host, "metrics not reachable:", e)
else:
    print("local mode: see the loopback telemetry table above")